# Ground Classification: CSF vs PMF

Separating ground from above-ground returns is the foundational step in most
LiDAR workflows.  Occulus provides two algorithms:

| Algorithm | Best for | Key parameters |
|---|---|---|
| **CSF** — Cloth Simulation Filter | Natural terrain, forests | `cloth_resolution`, `class_threshold` |
| **PMF** — Progressive Morphological Filter | Urban, mixed terrain | `max_window_size`, `slope` |

This notebook compares both on synthetic terrain with buildings, trees, and flat ground.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from occulus.types import AerialCloud
from occulus.segmentation import classify_ground_csf, classify_ground_pmf

rng = np.random.default_rng(42)

def make_synthetic_scene(rng, n=40_000):
    """Generate a synthetic urban/rural mixed scene."""
    x = rng.uniform(0, 200, n)
    y = rng.uniform(0, 200, n)
    # Gentle rolling terrain
    z = 5 * np.sin(x / 40) + 3 * np.cos(y / 30) + rng.normal(0, 0.05, n)
    
    # Building 1: 20x15 m, 12 m tall
    b1 = (x > 30) & (x < 50) & (y > 40) & (y < 55)
    z[b1] += rng.uniform(0, 12, b1.sum())
    
    # Building 2: 25x20 m, 8 m tall
    b2 = (x > 120) & (x < 145) & (y > 100) & (y < 120)
    z[b2] += rng.uniform(0, 8, b2.sum())
    
    # Trees: 15 random cylinders, 3-15 m tall
    for _ in range(15):
        tx, ty = rng.uniform(10, 190), rng.uniform(10, 190)
        th = rng.uniform(3, 15)
        t_mask = np.hypot(x - tx, y - ty) < 3
        z[t_mask] += rng.uniform(0, th, t_mask.sum())
    
    return np.column_stack([x, y, z])

xyz = make_synthetic_scene(rng)
cloud = AerialCloud(xyz.astype(np.float64))
print(f'Scene: {cloud.n_points:,} points, Z range {xyz[:,2].min():.1f}–{xyz[:,2].max():.1f} m')

## CSF Ground Classification

In [ ]:
import time

t0 = time.perf_counter()
csf_result = classify_ground_csf(cloud)
csf_time = time.perf_counter() - t0

csf_ground = csf_result.classification == 2
print(f'CSF  — ground: {csf_ground.sum():,} ({csf_ground.mean():.1%})  [{csf_time:.2f}s]')

## PMF Ground Classification

In [ ]:
t0 = time.perf_counter()
pmf_result = classify_ground_pmf(cloud)
pmf_time = time.perf_counter() - t0

pmf_ground = pmf_result.classification == 2
print(f'PMF  — ground: {pmf_ground.sum():,} ({pmf_ground.mean():.1%})  [{pmf_time:.2f}s]')

# Agreement rate
agreement = (csf_ground == pmf_ground).mean()
print(f'Agreement between CSF and PMF: {agreement:.2%}')

## Side-by-Side Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

def scatter_class(ax, xyz, mask, title):
    ax.scatter(xyz[mask, 0], xyz[mask, 1], s=0.5, c='sienna', label='Ground', alpha=0.6)
    ax.scatter(xyz[~mask, 0], xyz[~mask, 1], s=0.5, c='forestgreen', label='Above-ground', alpha=0.4)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.legend(markerscale=5, fontsize=8)

scatter_class(axes[0], xyz, csf_ground, f'CSF ({csf_ground.mean():.1%} ground)')
scatter_class(axes[1], xyz, pmf_ground, f'PMF ({pmf_ground.mean():.1%} ground)')

# Disagreement map
disagree = csf_ground != pmf_ground
axes[2].scatter(xyz[~disagree, 0], xyz[~disagree, 1], s=0.5, c='lightgray', alpha=0.3)
axes[2].scatter(xyz[disagree, 0], xyz[disagree, 1], s=1.5, c='red', alpha=0.8)
axes[2].set_title(f'Disagreements ({disagree.mean():.1%})', fontsize=11)
axes[2].set_xlabel('X (m)'); axes[2].set_ylabel('Y (m)')

plt.suptitle('Ground Classification Comparison: CSF vs PMF', fontsize=13)
plt.tight_layout()
plt.savefig('../../outputs/ground_classification_comparison.png', dpi=150)
plt.show()

## When to Use Which?

| Scenario | Recommendation |
|---|---|
| Dense forest, natural terrain | **CSF** |
| Urban/suburban with buildings | **PMF** |
| Steep mountain terrain | **CSF** with small `cloth_resolution` |
| Very flat agricultural land | Either — PMF is faster |
| Mixed scene with both | Run both, take intersection (conservative) |

**Next**: `04_registration_workflow.ipynb`